# Diffusion in an isotropic 3D harmonic well
The model<sup>1</sup> describes a particle that diffuses with diffusion coefficient $D$ in a harmonic potential that is the same in all three directions, so its position about the centre of the well has a Gaussian distribution. $\langle u^2 \rangle$ is the mean square distance from the centre, summed over the three directions. Because the well is isotropic, the scattering depends only on the magnitude of $Q$:

$$ S(Q, E) = A_0(Q)\,\delta(E) + \sum_{i=1}^{n} A_i(Q)\, L(E, \Gamma_i) $$

with

$$ A_i(Q) = e^{-y} \frac{y^i}{i!}, \quad y = \frac{Q^2 \langle u^2 \rangle}{3},
\quad \Gamma_i = \frac{3 i \hbar D}{\langle u^2 \rangle}. $$

$A_0 = e^{-y}$ is the Elastic Incoherent Structure Factor (EISF) and $A_i$ the Quasi-Elastic Incoherent Structure Factor (QISF) of order $i$. The paper<sup>1</sup> uses the variance along one direction, $\langle u_x^2 \rangle = \langle u^2 \rangle / 3$.

The QISF form a Poisson distribution in $i$ with mean $y$, and together with the EISF they sum to one. The infinite sum is therefore truncated at $n$ = `number_of_lorentzians`, which must be well above $y$ at the largest $Q$.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

import easydynamics as edyn

%matplotlib widget

## The model at different Q values

In [ ]:
# Q is in Angstrom^-1, D in m^2/s and <u^2> in Angstrom^2.
Q = np.linspace(0.5, 2, 7)

gaussian_model = edyn.GaussianModel3D(
    scale=1.0,
    diffusion_coefficient=2e-9,
    mean_u_squared=1.5,
    number_of_lorentzians=20,
    Q=Q,
)
print(gaussian_model)

In [ ]:
# Plot the quasi-elastic part (all Lorentzians) at every Q. The delta function
# would be a spike at E = 0; its weight, the EISF, is plotted further down.
energy = np.linspace(-2, 2, 501)
component_collections = gaussian_model.get_component_collections()

cmap = plt.cm.jet
nQ = len(component_collections)
plt.figure()
for Q_index, collection in enumerate(component_collections):
    quasielastic = sum(
        component.evaluate(energy)
        for component in collection
        if isinstance(component, edyn.Lorentzian)
    )
    color = cmap(Q_index / (nQ - 1))
    plt.plot(energy, quasielastic, label=f'Q={Q[Q_index]:.2f} Å^-1', color=color)

plt.legend()
plt.xlabel('Energy (meV)')
plt.ylabel('Intensity (arb. units)')
plt.title('3D Gaussian model')
plt.show()

## Widths and weights

The widths grow linearly with the order and do not depend on $Q$. The weights do: at small $Q$ almost all weight is in the elastic line and the first Lorentzians, while at large $Q$ it moves to higher orders.

In [ ]:
orders = np.arange(1, gaussian_model.number_of_lorentzians + 1)
widths = [gaussian_model.calculate_width(order=order)[0] for order in orders]
print(f'HWHM of order 1: {widths[0]:.4f} meV, of order 20: {widths[-1]:.4f} meV')

plt.figure()
for Q_value in [0.5, 1.5, 3.0]:
    weights = [gaussian_model.calculate_QISF_order(Q_value, order=order)[0] for order in orders]
    plt.plot(orders, weights, marker='o', label=f'Q={Q_value} Å^-1')
plt.xlabel('Order i')
plt.ylabel('QISF of order i, $A_i$')
plt.title('QISF $A_i$ of each Lorentzian (a Poisson distribution in i)')
plt.legend()
plt.show()

## EISF and QISF as a function of Q

The EISF, $\exp(-Q^2 \langle u^2 \rangle / 3)$, is the same Debye-Waller factor as in `DeltaLorentz`.

In [ ]:
Q_fine = np.linspace(0.01, 4, 200)
delta_lorentz = edyn.DeltaLorentz(mean_u_squared=1.5)

plt.figure()
plt.plot(Q_fine, gaussian_model.calculate_EISF(Q_fine), label='EISF')
plt.plot(Q_fine, gaussian_model.calculate_QISF(Q_fine), label='QISF')
plt.plot(
    Q_fine,
    delta_lorentz.calculate_EISF(Q_fine),
    color='k',
    linestyle='dotted',
    label='DeltaLorentz EISF',
)
plt.xlabel('Q (Å$^{-1}$)')
plt.ylabel('Weight')
plt.title('EISF and QISF of the 3D Gaussian model')
plt.legend()
plt.show()

## Truncation

Truncating the sum leaves out the weight of the orders above $n$. A warning is raised when that missing weight exceeds $10^{-3}$ at any of the model's $Q$ values, and `calculate_missing_weight` returns it. As a rule, $n$ must be well above $y = Q^2 \langle u^2 \rangle / 3$ at the largest $Q$.

In [ ]:
Q_wide = np.linspace(0.5, 3, 6)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    truncated_model = edyn.GaussianModel3D(
        diffusion_coefficient=2e-9, mean_u_squared=1.5, number_of_lorentzians=5, Q=Q_wide
    )
for warning in caught:
    print(warning.message)

print('Missing weight per Q:', np.round(truncated_model.calculate_missing_weight(), 4))
largest = gaussian_model.calculate_missing_weight(Q_wide).max()
print(f'With 20 Lorentzians the largest missing weight is {largest:.1e}')

## References

1. F. Volino, J.-C. Perrin and S. Lyonnard, J. Phys. Chem. B **110**, 11217 (2006), [doi:10.1021/jp061103s](https://doi.org/10.1021/jp061103s).